DGM a cui passo i 3 archi e le features dei nodi. In questo caso prima che i dati arrivano al DGM, si usa uno strato convolutivo che legge le feature fisiche di tutti e tre gli archi e le inserisce matematicamente dentro i vettori dei nodi. 

In [6]:
# Installa PyTorch Geometric
!pip install torch_geometric


In [7]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, GCNConv, GATv2Conv
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_fscore_support,
    accuracy_score,
    confusion_matrix,
    roc_curve
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Cella modificata con l'aggiunta del DGM:

In [ ]:
# ==========================================
# 1) Moduli DGM (Prelevati dal Prof)
# ==========================================
class MLP(nn.Module):
    """Feed-forward network per creare lo spazio latente."""
    def __init__(self, dims, dropout=0.0):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers.append(nn.ReLU())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class DynamicGraphModule(nn.Module):
    """Generatore di archi latenti dinamici (kNN basato sulle feature)."""
    def __init__(self, input_dim, hidden_dim=32, latent_dim=16, k=5):
        super().__init__()
        self.embed = MLP([2 * input_dim, hidden_dim, latent_dim])
        self.k = int(k)
        self.temperature = nn.Parameter(torch.tensor(1.0))
        
    @staticmethod
    def aggregate_initial_neighbors(x, edge_index):
        if edge_index is None:
            return torch.zeros_like(x)
        src, dst = edge_index
        out = torch.zeros_like(x)
        deg = torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)
        out.index_add_(0, src, x[dst])
        deg.index_add_(0, src, torch.ones_like(src, dtype=x.dtype))
        return out / deg.clamp_min(1.0).unsqueeze(-1)
        
    def forward(self, x, initial_edge_index=None):
        # 1. Contesto iniziale basato sul grafo originale
        context = self.aggregate_initial_neighbors(x, initial_edge_index)
        
        # 2. Embedding nello spazio latente
        z = self.embed(torch.cat([x, context], dim=-1))

        # 3. Calcolo delle distanze e probabilità di collegamento
        diag_mask = torch.eye(x.shape[0], dtype=torch.bool, device=x.device)
        dist = torch.cdist(z, z, p=2).pow(2)
        scale = torch.exp(self.temperature.clamp(-5.0, 5.0))
        logits = -dist * scale
        logits = logits.masked_fill(diag_mask, -1e9)

        probs = torch.softmax(logits, dim=1)
        probs = probs.masked_fill(diag_mask, 0.0)

        # 4. Estrazione dei k-vicini per creare i nuovi archi
        k = min(max(1, self.k), x.shape[0] - 1)
        idx = torch.topk(probs, k=k, dim=1, largest=True).indices
        src = torch.arange(x.shape[0], device=x.device).view(-1, 1).expand_as(idx).reshape(-1)
        dst = idx.reshape(-1)
        edge_index = torch.stack([src, dst], dim=0)

        return z, edge_index, probs

# ==========================================
# 2) HeteroEdgeClassifier (Contaminato con Pre-DGM e Imbuto Cieco)
# ==========================================
class HeteroEdgeClassifier(nn.Module):
    def __init__(self, in_channels_ip, hidden_channels=64, out_channels=32, num_classes=2, dgm_k=2):
        super().__init__()
        
        # 1. FASE PRE-DGM: Leggiamo le feature fisiche di tutti e 3 gli archi
        self.pre_conv = HeteroConv({
            ('ip', 'network', 'ip'): GATv2Conv(-1, hidden_channels, edge_dim=2, add_self_loops=False),
            ('ip', 'context', 'ip'): GATv2Conv(-1, hidden_channels, edge_dim=2, add_self_loops=False),
            ('ip', 'knowledge', 'ip'): GATv2Conv(-1, hidden_channels, edge_dim=2, add_self_loops=False)
        }, aggr='sum')
        
        self.pre_bn = nn.BatchNorm1d(hidden_channels)

        # 2. MOTORE DGM: Prende i nodi arricchiti da tutti i contesti
        self.dgm = DynamicGraphModule(
            input_dim=hidden_channels, 
            hidden_dim=hidden_channels, 
            latent_dim=16, 
            k=dgm_k
        )

        # 3. FASE POST-DGM: "L'Imbuto Cieco". Lavora SOLO sull'arco latent
        self.post_conv1 = GCNConv(16, hidden_channels) 
        self.post_bn1 = nn.BatchNorm1d(hidden_channels)
        
        self.post_conv2 = GCNConv(hidden_channels, hidden_channels)
        self.post_bn2 = nn.BatchNorm1d(hidden_channels)

        self.lin_node = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels * 2),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(hidden_channels * 2, out_channels),
            nn.ReLU()
        )

        # 4. CLASSIFICATORE LINEARE
        self.lin_edge = nn.Sequential(
            nn.Linear(out_channels * 2, out_channels),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(out_channels, out_channels),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(out_channels, num_classes)
        )

    def forward(self, data):
        device = data.x_dict['ip'].device
        
        # Recuperiamo la mappa delle 3 topologie originali
        edge_net = data.edge_index_dict.get(('ip', 'network', 'ip'), torch.empty((2, 0), dtype=torch.long, device=device))
        edge_ctx = data.edge_index_dict.get(('ip', 'context', 'ip'), torch.empty((2, 0), dtype=torch.long, device=device))
        edge_knw = data.edge_index_dict.get(('ip', 'knowledge', 'ip'), torch.empty((2, 0), dtype=torch.long, device=device))

        clean_edge_index = {
            ('ip', 'network', 'ip'): edge_net,
            ('ip', 'context', 'ip'): edge_ctx,
            ('ip', 'knowledge', 'ip'): edge_knw
        }
        
        clean_edge_attr = {
            ('ip', 'network', 'ip'): data['ip', 'network', 'ip'].edge_attr,
            ('ip', 'context', 'ip'): data['ip', 'context', 'ip'].edge_attr,
            ('ip', 'knowledge', 'ip'): data['ip', 'knowledge', 'ip'].edge_attr,
        }
        
        # Le feature fisiche si fondono nei vettori dei nodi IP
        x_dict = self.pre_conv(data.x_dict, clean_edge_index, edge_attr_dict=clean_edge_attr)
        x_enriched = self.pre_bn(x_dict['ip'])
        x_enriched = F.relu(x_enriched)

        # Concateniamo tutte e 3 le topologie per creare la mappa iniziale
        initial_edges_aggregated = torch.cat([edge_net, edge_ctx, edge_knw], dim=1)

        # Il DGM riceve i nodi arricchiti e la la topologia iniziale
        z_latent, learned_edge_index, _ = self.dgm(x_enriched, initial_edges_aggregated)        
       

        # POST-CONVOLUZIONE: Passiamo SOLO lo spazio latente del DGM e i suoi nuovi archi
        out_nodes = self.post_conv1(z_latent, learned_edge_index)
        out_nodes = self.post_bn1(out_nodes)
        out_nodes = F.relu(out_nodes)
        
        out_nodes = self.post_conv2(out_nodes, learned_edge_index)
        out_nodes = self.post_bn2(out_nodes)
        out_nodes = F.relu(out_nodes)

        out_nodes = self.lin_node(out_nodes)

        return {'ip': out_nodes}

    def edge_classify(self, z_src, z_dst):
        z_cat = torch.cat([z_src, z_dst], dim=-1)
        return self.lin_edge(z_cat)


# ==========================================
# 3) Focal Loss (Corretta per il grave sbilanciamento)
# ==========================================
def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    # Calcolo dell'errore base
    ce_loss = F.cross_entropy(logits, targets, reduction='none')
    pt = torch.exp(-ce_loss)

    #alpha: penalizza maggiormente gli errori sulla classe minoritaria, ovvero gli attacchi (1)
    # per compensare il grave sbilanciamento del dataset
    alpha_t = torch.where(
        targets == 1, 
        torch.tensor(alpha, device=targets.device), 
        torch.tensor(1.0 - alpha, device=targets.device)
    )

    return (alpha_t * (1 - pt)**gamma * ce_loss).mean()

In [9]:
# ==========================================
# 4) ADDESTRAMENTO
# ==========================================
def train_edge_classifier(model, graphs, epochs=10, device='cuda', alpha=0.25, gamma=2.0):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for data in graphs:
            data = data.to(device)
            optimizer.zero_grad()

            x_dict = model(data)
            z_ip = x_dict['ip']

            loss_graph = 0.0
            
            # Valutiamo su tutti gli archi (network, context, knowledge)
            for rel_type in data.edge_types:
                if 'edge_label' not in data[rel_type]: continue
                labels = data[rel_type].edge_label
                if labels.numel() == 0: continue

                src, dst = data[rel_type].edge_index
                logits = model.edge_classify(z_ip[src], z_ip[dst])

                loss_graph += focal_loss(logits, labels, alpha=alpha, gamma=gamma)

            if isinstance(loss_graph, torch.Tensor):
                loss_graph.backward()
                optimizer.step()
                total_loss += loss_graph.item()

        print(f"[Epoch {epoch}/{epochs}] Loss: {total_loss / len(graphs):.4f}")

In [ ]:
%matplotlib inline
# ==========================================
# 5) VALUTAZIONE CON MASSIMIZZAZIONE F1-SCORE
# ==========================================
@torch.no_grad()
def evaluate_edge_classifier(model, test_graphs, device='cuda'):
    
    from sklearn.metrics import precision_recall_curve, roc_curve, roc_auc_score, precision_recall_fscore_support, accuracy_score, confusion_matrix
    import numpy as np
    
    model.eval()
    model.to(device)

    all_scores, all_labels = [], []

    for data in test_graphs:
        data = data.to(device)
        x_dict = model(data)
        z_ip = x_dict['ip']

        # Valutiamo su tutti gli archi (network, context, knowledge)
        for rel_type in data.edge_types:
            if 'edge_label' not in data[rel_type]: continue
            labels = data[rel_type].edge_label
            if labels.numel() == 0: continue

            src, dst = data[rel_type].edge_index
            logits = model.edge_classify(z_ip[src], z_ip[dst])

            probs = F.softmax(logits, dim=-1)[:, 1]
            all_scores.extend(probs.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)

    if len(np.unique(all_labels)) < 2:
        print("Only one class in test set => AUC=0.5 by definition.")
        return

    # SOGLIA DINAMICA (Indice di Youden sulla Curva ROC)
    fpr, tpr, thresholds_roc = roc_curve(all_labels, all_scores)
    
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds_roc[optimal_idx]

    print("\n" + "="*50)
    print(f"🎯 SOGLIA OTTIMALE SCOPERTA (Max Youden's J): {optimal_threshold:.6f}")
    print("="*50 + "\n")

    auc = roc_auc_score(all_labels, all_scores)
    final_preds = (all_scores > optimal_threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, final_preds, average='binary', zero_division=0
    )
    accuracy = accuracy_score(all_labels, final_preds)
    cm = confusion_matrix(all_labels, final_preds)

    print(f"AUC={auc:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}, Accuracy={accuracy:.4f}")
    print("Confusion Matrix:\n", cm)



# 6) Data-Loading Helpers (Normalizzazione Completa)
###############################
def load_single_graph(directory, filename):
    path = os.path.join(directory, filename)
    data = torch.load(path, weights_only=False)

    # 1. NORMALIZZAZIONE NODI (Cruciale per in_degree / out_degree immensi)
    if 'ip' in data.x_dict:
        data.x_dict['ip'] = torch.log1p(torch.abs(data.x_dict['ip']))

    for rel_type in data.edge_types:
        # 2. SISTEMAZIONE LABEL E CAST A LONG (Intero per la CrossEntropy)
        if 'edge_label' in data[rel_type]:
            lbl = data[rel_type].edge_label
            lbl = torch.where(lbl < 0, torch.tensor(0, device=lbl.device), lbl)
            lbl = torch.where(lbl > 1, torch.tensor(1, device=lbl.device), lbl)
            data[rel_type].edge_label = lbl.long()
            
        # 3. NORMALIZZAZIONE ARCHI 
        if 'edge_attr' in data[rel_type]:
            attr = data[rel_type].edge_attr
            data[rel_type].edge_attr = torch.log1p(torch.abs(attr))

    return data



# ==========================================
# 6) MAIN SCRIPT KAGGLE
# ==========================================
if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # Assicurati che questi path puntino ai nuovi dati estratti con le feature modificate
    train_dir = "/kaggle/input/datasets/irenelontano/grafitesi/train_hetero_graphs_pt"
    test_dir  = "/kaggle/input/datasets/irenelontano/grafitesi/test_hetero_graphs_pt"

    train_files = [f for f in os.listdir(train_dir) if f.endswith('.pt')]
    train_graphs = [load_single_graph(train_dir, f) for f in train_files]
    print(f"Loaded {len(train_graphs)} training graphs.")

    test_files = [f for f in os.listdir(test_dir) if f.endswith('.pt')]
    test_graphs = [load_single_graph(test_dir, f) for f in test_files]
    print(f"Loaded {len(test_graphs)} test graphs.")

    # in_channels_ip a 4 (Community, In-Degree, Out-Degree, Centrality)
    model = HeteroEdgeClassifier(in_channels_ip=4, hidden_channels=64, out_channels=32, dgm_k=5)

    print("\n--- Inizio Addestramento ---")
    train_edge_classifier(model, train_graphs, epochs=70, device=device)

    print("\n--- Inizio Valutazione sul Test Set ---")
    evaluate_edge_classifier(model, test_graphs, device=device)

Using device: cuda
Loaded 74 training graphs.
Loaded 19 test graphs.

--- Inizio Addestramento ---
[Epoch 1/70] Loss: 0.1652
[Epoch 2/70] Loss: 0.0048
[Epoch 3/70] Loss: 0.0044
[Epoch 4/70] Loss: 0.0042
[Epoch 5/70] Loss: 0.0039
[Epoch 6/70] Loss: 0.0036
[Epoch 7/70] Loss: 0.0042
[Epoch 8/70] Loss: 0.0036
[Epoch 9/70] Loss: 0.0033
[Epoch 10/70] Loss: 0.0027
[Epoch 11/70] Loss: 0.0031
[Epoch 12/70] Loss: 0.0028
[Epoch 13/70] Loss: 0.0029
[Epoch 14/70] Loss: 0.0029
[Epoch 15/70] Loss: 0.0027
[Epoch 16/70] Loss: 0.0027
[Epoch 17/70] Loss: 0.0025
[Epoch 18/70] Loss: 0.0029
[Epoch 19/70] Loss: 0.0034
[Epoch 20/70] Loss: 0.0032
[Epoch 21/70] Loss: 0.0028
[Epoch 22/70] Loss: 0.0029
[Epoch 23/70] Loss: 0.0032
[Epoch 24/70] Loss: 0.0032
[Epoch 25/70] Loss: 0.0033
[Epoch 26/70] Loss: 0.0030
[Epoch 27/70] Loss: 0.0029
[Epoch 28/70] Loss: 0.0032
[Epoch 29/70] Loss: 0.0033
[Epoch 30/70] Loss: 0.0033
[Epoch 31/70] Loss: 0.0037
[Epoch 32/70] Loss: 0.0035
[Epoch 33/70] Loss: 0.0031
[Epoch 34/70] Loss: